# Adapta el conciliador a tu banco

**IMEF · Comité Técnico Nacional de Transformación y Economía Digital**
Automatización de Procesos Financieros · Benjamín Oliva Vázquez · 13 de agosto de 2026

---

En la sesión vimos un motor que concilia 226 movimientos en menos de un milisegundo.
Tiene un problema evidente: espera columnas llamadas `fecha`, `referencia`,
`descripcion`, `cargo` y `abono`. **Tu estado de cuenta no se llama así.**

Ese pegamento —leer el archivo de *tu* banco y traducirlo al formato que el motor
entiende— es trabajo real, aburrido y distinto para cada institución. Es exactamente
la clase de tarea donde conviene que la IA escriba el código.

Lo que vas a hacer aquí:

1. Mirar un estado de cuenta con formato **ajeno** al del motor.
2. Pedirle al modelo que escriba el adaptador.
3. **Leer el código que escribió** antes de ejecutarlo.
4. Correr el mismo motor de siempre sobre el archivo traducido.

> El motor no se toca. Lo que la IA escribe es el pegamento, y el pegamento se revisa.


## 1 · Preparar el entorno

Se clona el repositorio público de la sesión e instala el SDK.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = "imef-automatizacion-procesos"
SENAL = Path("demos") / "shared" / "motor.py"


def buscar_raiz() -> Path | None:
    """Busca el repo hacia arriba desde el directorio actual y un nivel abajo."""
    aqui = Path.cwd().resolve()
    for p in [aqui, *aqui.parents]:
        if (p / SENAL).exists():
            return p
        if (p / REPO / SENAL).exists():
            return p / REPO
    return None


raiz = buscar_raiz()
if raiz is None:
    subprocess.run(["git", "clone", "-q",
                    f"https://github.com/benjov/{REPO}.git"], check=True)
    raiz = Path(REPO).resolve()

os.chdir(raiz)
sys.path.insert(0, str(raiz / "demos"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
print("Listo. Trabajando en:", raiz)

## 2 · Tu llave

En Colab: **🔑 Secrets** (panel izquierdo) → agrega `ANTHROPIC_API_KEY` y activa el
acceso para este cuaderno. Si no, se pide aquí y no queda escrita en el archivo.

In [ ]:
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

import anthropic
cliente = anthropic.Anthropic()
MODELO = "claude-opus-5"
print("Cliente listo.")

## 3 · El archivo que el motor NO entiende

Este es el mismo mes de operaciones, tal como lo entregaría otra institución:
encabezados en mayúsculas, fecha en `dd/mm/aaaa`, importes con separador de miles
y columnas `RETIRO`/`DEPOSITO` en lugar de `cargo`/`abono`.

In [ ]:
AJENO = Path("demos/data/estado_cuenta_otro_banco.csv")

muestra = "".join(AJENO.read_text(encoding="utf-8").splitlines(True)[:6])
print(muestra)

## 4 · Que la IA escriba el adaptador

No se le pide "concilia esto". Se le pide una función con un contrato preciso:
recibe una ruta, devuelve un `DataFrame` con las cinco columnas canónicas.

Un contrato estrecho es lo que hace verificable el resultado: se puede leer en
treinta segundos y probar en uno.

In [ ]:
FORMATO_CANONICO = """fecha        texto ISO aaaa-mm-dd
referencia   texto
descripcion  texto
cargo        número, salida de dinero (0 si no aplica)
abono        número, entrada de dinero (0 si no aplica)"""

prompt = f"""Escribe una función de Python con esta firma exacta:

    def adaptar(ruta: str) -> "pandas.DataFrame"

Recibe la ruta de un CSV de estado de cuenta bancario y devuelve un DataFrame
con EXACTAMENTE estas columnas, en este orden:

{FORMATO_CANONICO}

Estas son las primeras líneas del archivo real:

{muestra}

Requisitos:
- Usa sólo pandas y la librería estándar.
- Convierte la fecha al formato ISO; el archivo viene en dd/mm/aaaa.
- Los importes traen separador de miles y pueden venir vacíos: conviértelos a
  número, con 0 donde no haya valor.
- No imprimas nada ni ejecutes la función.

Devuelve ÚNICAMENTE el código de la función, sin explicación y sin cercas de
código markdown."""

# Dos precauciones que no son decorativas (ver la nota de abajo):
#   · max_tokens acota TODA la respuesta, razonamiento incluido. Corto, el
#     modelo puede gastarlo entero pensando y devolver cero bloques de texto.
#   · fallbacks="default" rescata la llamada si un clasificador de seguridad
#     la rechaza por error.
r = cliente.beta.messages.create(
    model=MODELO, max_tokens=8000,
    betas=["server-side-fallback-2026-07-01"],
    fallbacks="default",
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": prompt}],
)

for b in r.content:
    if b.type == "fallback":
        print(f"[aviso] {b.from_.model} declinó; respondió {b.to.model}\n")

textos = [b.text for b in r.content if b.type == "text"]
if not textos:
    raise RuntimeError(
        f"La respuesta no trae texto (stop_reason={r.stop_reason}). "
        "Si dice 'max_tokens', sube el límite; si dice 'refusal', vuelve a "
        "intentar."
    )

codigo = textos[0].strip()
codigo = codigo.removeprefix("```python").removeprefix("```").removesuffix("```").strip()

print(codigo)

> **Por qué esta celda trae dos cinturones de seguridad**
>
> Al preparar este cuaderno, esta misma petición —escribir un lector de CSV— fue
> **rechazada por el clasificador de seguridad del modelo**, con categoría
> *cyber*. Un falso positivo: código contable inocuo que se parecía lo
> suficiente a algo que no lo es. Reintentada, pasó. El rechazo es
> intermitente, no determinista.
>
> Eso es información útil para quien va a poner esto en un proceso real: los
> modelos frontera traen clasificadores, los clasificadores se equivocan en
> ambas direcciones, y un proceso de cierre no puede detenerse porque una
> llamada volvió vacía. La respuesta de ingeniería es la de arriba —
> `fallbacks="default"`, que reencamina la petición a otro modelo dentro de la
> misma llamada— más la comprobación explícita de que sí llegó texto.
>
> Regla general: **nunca leas `r.content[0]` sin revisar antes `stop_reason`.**

## 5 · Léelo antes de ejecutarlo

Arriba está el código completo. **Léelo.** No es una formalidad: la siguiente celda
lo ejecuta en tu sesión.

Tres preguntas que bastan para este caso: ¿abre sólo el archivo que le pasaste?
¿hace únicamente la conversión de formato? ¿no manda nada a ninguna parte?

Esta es la parte del "vibe coding" que suele omitirse en las demos, y es la única
que separa un atajo útil de un problema nuevo. La IA escribe el borrador; la
responsabilidad de lo que corre en tu máquina sigue siendo tuya.

In [ ]:
import pandas as pd

espacio = {}
exec(codigo, espacio)          # revisado en la celda anterior
adaptar = espacio["adaptar"]

df = adaptar(str(AJENO))
print(f"{len(df)} movimientos traducidos")
print("Columnas:", list(df.columns))
df.head()

## 6 · El mismo motor, sin tocarle una línea

In [ ]:
from shared.motor import cargar_banco, cargar_libros, conciliar

TRADUCIDO = Path("demos/data/_traducido.csv")
df.assign(saldo=0).to_csv(TRADUCIDO, index=False)

banco = cargar_banco(TRADUCIDO)
libros = cargar_libros(Path("demos/data/auxiliar_contable.csv"))
res = conciliar(banco, libros)

print(f"Conciliados : {res.conciliados}/{res.total_movimientos} ({res.tasa:.1%})")
print(f"Tiempo      : {res.segundos * 1000:.2f} ms")
print(f"Excepciones : {len(res.excepciones)}  por ${res.monto_en_excepcion:,.2f}")
print()
for m in res.excepciones:
    print(f"  {m.id}  {m.fecha}  {m.monto:>13,.2f}   {m.descripcion[:44]}")

## 7 · La comprobación que importa

El resultado debe ser idéntico al del archivo original. Si la traducción hubiera
perdido o deformado un movimiento, aquí se nota.

In [ ]:
original = conciliar(cargar_banco(Path("demos/data/estado_cuenta.csv")), libros)

igual = (res.conciliados == original.conciliados
         and len(res.excepciones) == len(original.excepciones)
         and round(res.monto_en_excepcion, 2) == round(original.monto_en_excepcion, 2))

print("Original :", original.conciliados, "conciliados,",
      len(original.excepciones), "excepciones")
print("Traducido:", res.conciliados, "conciliados,", len(res.excepciones), "excepciones")
print()
print("✅ La traducción es fiel." if igual else
      "❌ Algo se perdió en la traducción: revisa el adaptador.")

---

## Lo que acaba de pasar

La IA no concilió nada. Escribió **veinte líneas de pegamento** que nadie quería
escribir, y el motor determinista —el mismo de la sesión, sin una línea
modificada— hizo el trabajo exacto y lo hizo en un milisegundo.

Ese es el patrón que vale la pena llevarse:

| | Quién lo hace | Por qué |
|---|---|---|
| Traducir formatos, escribir pegamento | **La IA** | Cambia con cada banco, es aburrido, es verificable en un minuto |
| Cruzar, sumar, agrupar | **Código determinista** | Tiene que ser exacto y auditable |
| Explicar la excepción rara | **La IA** | Requiere leer lenguaje humano y política |
| Autorizar el asiento | **Una persona** | Alguien firma, y no se delega |

### Pruébalo con tu propio archivo

Sube tu estado de cuenta a Colab, cambia `AJENO` por su ruta y vuelve a correr desde
la celda 4. El prompt no cambia: describe el contrato, no el banco.

---

Repositorio: <https://github.com/benjov/imef-automatizacion-procesos>
Contacto: <benjamin@analiticaboutique.com.mx>